# ML Data Preparation and Final Validation

This notebook prepares the selected Hotel Booking features for the later
machine-learning stage. The input is
`data/processed/hotel_booking_features.csv`, produced during feature engineering.
The target variable is `is_canceled`, where 0 means the booking was not
canceled and 1 means it was canceled.

The prediction point assumed by the group is when the initial reservation is
created. The workflow separates predictors and target, creates a stratified
training/test split, fits preprocessing using training data only, transforms
both partitions consistently, and validates the resulting data.


### 1. Load and Review the Selected Feature Dataset

This section loads the dataset produced by feature engineering and the list of
features selected by Manesha. It checks that the dataset contains the expected
target and predictor columns before any train/test split or transformation.

The categorical ID fields `agent` and `company` are read as text because their
values represent identifiers, not numerical quantities.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

# Find the project root whether the notebook is run from the project folder
# or from 05_ML_Data_Preparation.
ROOT = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "data/processed/hotel_booking_features.csv").is_file()
)

DATA_PATH = ROOT / "data/processed/hotel_booking_features.csv"
FEATURE_LIST_PATH = (
    ROOT / "04_Feature_Engineering/final_selected_features.csv"
)


df = pd.read_csv(
    DATA_PATH,
    dtype={"agent": "string", "company": "string", "country": "string"}
)

feature_list = pd.read_csv(FEATURE_LIST_PATH)
selected_features = feature_list["selected_feature"].tolist()

print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Number of selected predictors: {len(selected_features)}")
print(f"Target column present: {'is_canceled' in df.columns}")

display(feature_list)
display(df.head())

Dataset shape: 119,205 rows × 33 columns
Number of selected predictors: 32
Target column present: True


,selected_feature
0,hotel
1,lead_time
2,arrival_date_year
3,arrival_date_month
4,arrival_date_week_number
5,arrival_date_day_of_month
6,stays_in_weekend_nights
7,stays_in_week_nights
8,adults
9,children


,is_canceled,hotel,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,customer_type,adr,required_car_parking_spaces,total_of_special_requests,total_stay,total_guests,has_children,has_babies,previous_total_bookings,previous_cancellation_rate
0,0,Resort Hotel,342,2015,July,27,1,0,0,2,...,Transient,0.0,0,0,0,2,0,0,0,0.0
1,0,Resort Hotel,737,2015,July,27,1,0,0,2,...,Transient,0.0,0,0,0,2,0,0,0,0.0
2,0,Resort Hotel,7,2015,July,27,1,0,1,1,...,Transient,75.0,0,0,1,1,0,0,0,0.0
3,0,Resort Hotel,13,2015,July,27,1,0,1,1,...,Transient,75.0,0,0,1,1,0,0,0,0.0
4,0,Resort Hotel,14,2015,July,27,1,0,2,2,...,Transient,98.0,0,1,2,2,0,0,0,0.0


### 2. Separate the Target and Predictors

The target (`y`) is `is_canceled`: 0 means not canceled and 1 means canceled.
The predictor table (`X`) contains only the 32 features selected in the
feature-engineering stage.

Selecting X from the documented feature list also prevents the target from
accidentally being included as a predictor. No rows or target labels are
changed in this step.

In [2]:
# Confirm that the selected feature list matches the dataset.
assert len(selected_features) == len(set(selected_features)), (
    "The selected feature list contains duplicate names."
)

missing_features = set(selected_features) - set(df.columns)
unexpected_columns = set(df.columns) - set(selected_features) - {"is_canceled"}

assert not missing_features, f"Selected features missing from dataset: {missing_features}"
assert not unexpected_columns, f"Unexpected dataset columns: {unexpected_columns}"
assert "is_canceled" not in selected_features, "The target must not be a predictor."

# Separate predictors and target without changing the original dataset.
X = df[selected_features].copy()
y = df["is_canceled"].copy()

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print("Target values:")
display(y.value_counts().sort_index().rename_axis("is_canceled").to_frame("bookings"))

X shape: (119205, 32)
y shape: (119205,)
Target values:


,bookings
is_canceled,
0,75010
1,44195


### 3. Create a Stratified Train/Test Split

We reserve 20% of the bookings for the final test set and use the other 80%
for training. `random_state=42` makes the split reproducible.

Stratifying by `y` keeps approximately the same cancellation proportion in
both sets. The test set will be used only for transformation and final model
evaluation; preprocessing methods that learn from data must be fitted on the
training set.

In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test:  {y_test.shape}")

class_counts = pd.DataFrame({
    "Full dataset": y.value_counts().sort_index(),
    "Training": y_train.value_counts().sort_index(),
    "Test": y_test.value_counts().sort_index()
})

class_percentages = class_counts.div(class_counts.sum(axis=0), axis=1) * 100

print("\nClass counts:")
display(class_counts)

print("Class percentages:")
display(class_percentages.round(2))

X_train: (95364, 32)
X_test:  (23841, 32)
y_train: (95364,)
y_test:  (23841,)

Class counts:


,Full dataset,Training,Test
is_canceled,,,
0,75010,60008,15002
1,44195,35356,8839


Class percentages:


,Full dataset,Training,Test
is_canceled,,,
0,62.93,62.93,62.93
1,37.07,37.07,37.07


### 4. Identify Feature Types

Different feature types need different preprocessing. Numerical measurements
may need scaling, binary indicators already contain 0/1 values, and nominal
categories need encoding.

The feature groups follow the encoding strategy documented in the Feature
Engineering stage rather than being inferred only from stored data types.
In particular, `agent` and `company` are categorical IDs even though their
values may look like numbers. This step defines column groups; it does not
fit any transformation.

In [4]:
STRATEGY_PATH = ROOT / "04_Feature_Engineering/encoding_strategy.csv"
encoding_strategy = pd.read_csv(STRATEGY_PATH)

# Check that every selected predictor has exactly one documented strategy.
assert encoding_strategy["column"].is_unique, (
    "A feature appears more than once in the encoding strategy."
)
assert set(encoding_strategy["column"]) == set(selected_features), (
    "The encoding strategy does not match the selected features."
)

numerical_features = encoding_strategy.loc[
    encoding_strategy["encoding_strategy"] == "numeric_passthrough_or_scale_later",
    "column"
].tolist()

binary_features = encoding_strategy.loc[
    encoding_strategy["encoding_strategy"] == "binary_passthrough",
    "column"
].tolist()

categorical_features = encoding_strategy.loc[
    encoding_strategy["encoding_strategy"] == "one_hot",
    "column"
].tolist()

high_cardinality_features = encoding_strategy.loc[
    encoding_strategy["encoding_strategy"] == "one_hot_with_rare_grouping",
    "column"
].tolist()

print(f"Numerical features: {len(numerical_features)}")
print(f"Binary features: {len(binary_features)}")
print(f"Categorical features: {len(categorical_features)}")
print(f"High-cardinality categorical features: {len(high_cardinality_features)}")

display(pd.DataFrame({
    "feature_group": [
        "Numerical",
        "Binary",
        "Categorical",
        "High-cardinality categorical"
    ],
    "features": [
        ", ".join(numerical_features),
        ", ".join(binary_features),
        ", ".join(categorical_features),
        ", ".join(high_cardinality_features)
    ]
}))

Numerical features: 18
Binary features: 3
Categorical features: 8
High-cardinality categorical features: 3


,feature_group,features
0,Numerical,"lead_time, arrival_date_year, arrival_date_wee..."
1,Binary,"is_repeated_guest, has_children, has_babies"
2,Categorical,"hotel, arrival_date_month, meal, market_segmen..."
3,High-cardinality categorical,"country, agent, company"


### 5. Encoding and Scaling Strategy

The preprocessing method depends on the meaning of each feature:

- Numerical features: use standard scaling so features with large ranges do
  not dominate distance-based or linear models. The scaler's mean and
  standard deviation must be learned from training data only.
- Binary features: keep the existing 0/1 values without scaling or encoding.
- Nominal categorical features: use one-hot encoding. Category names have no
  meaningful numerical order, so assigning them integer ranks would be
  misleading.
- High-cardinality categorical features (`country`, `agent`, `company`):
  group infrequent categories using frequencies learned from training data
  only, then one-hot encode them. This limits the number of output columns.
  Categories that appear only in the test set must not be used to fit the
  encoder.

The selected dataset is expected to have no missing values after the Data
Preprocessing stage. Missing values will be checked before fitting. No
encoder or scaler is fitted in this section.

### 6. Build the Preprocessing Pipeline

A `ColumnTransformer` applies a different transformation to each feature
group. Numerical features are scaled, binary features pass through unchanged,
and categorical features are one-hot encoded.

For high-cardinality features, categories occurring in fewer than 0.5% of
training rows will be grouped as infrequent when the pipeline is fitted.
The pipeline is defined here but is not fitted until the next section.

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Scaling statistics will be learned when this pipeline is fitted.
numerical_pipeline = Pipeline(steps=[
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numerical", numerical_pipeline, numerical_features),
        ("binary", "passthrough", binary_features),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "high_cardinality",
            OneHotEncoder(
                handle_unknown="infrequent_if_exist",
                min_frequency=0.005
            ),
            high_cardinality_features
        )
    ]
)

print("Preprocessing pipeline defined. It has not been fitted.")
display(preprocessor)

Preprocessing pipeline defined. It has not been fitted.


,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numerical', ...), ('binary', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``

### 7. Fit on Training Data and Transform Both Sets

Before fitting, check that the selected predictors contain no missing or
non-finite numerical values. The cleaned dataset is expected to pass these
checks; unexpected values should be investigated rather than silently filled.

Fit the preprocessing pipeline using `X_train` only. Then use that fitted
pipeline to transform `X_test`. This keeps scaling statistics and category
frequencies from being learned from the test set.

In [6]:
import numpy as np

# Validate the inputs before fitting.
assert not X_train.isna().any().any(), "Missing values found in X_train."
assert not X_test.isna().any().any(), "Missing values found in X_test."

assert np.isfinite(X_train[numerical_features + binary_features].to_numpy()).all(), (
    "Non-finite numerical values found in X_train."
)
assert np.isfinite(X_test[numerical_features + binary_features].to_numpy()).all(), (
    "Non-finite numerical values found in X_test."
)

# Learn scaling statistics and category groups from training rows only.
X_train_processed = preprocessor.fit_transform(X_train)

# Reuse the fitted transformations; do not call fit on the test set.
X_test_processed = preprocessor.transform(X_test)

print("Preprocessing fitted on X_train only.")
print(f"Processed training shape: {X_train_processed.shape}")
print(f"Processed test shape:     {X_test_processed.shape}")

Preprocessing fitted on X_train only.
Processed training shape: (95364, 116)
Processed test shape:     (23841, 116)


### 8. Validate the Processed Feature Data

Check that both transformed datasets have the expected number of rows, share
the same feature structure, and contain no missing or infinite numerical
values. Retrieve the transformed feature names to make the final model inputs
understandable.

The transformed data may be stored as sparse matrices because one-hot
encoding creates many zero values. Validation should not convert the full
matrices into dense tables unnecessarily.

In [7]:
from scipy import sparse

feature_names = preprocessor.get_feature_names_out()

def count_non_finite(matrix):
    """Count NaN and infinity without expanding a sparse matrix."""
    values = matrix.data if sparse.issparse(matrix) else np.asarray(matrix)
    return int((~np.isfinite(values)).sum())

train_non_finite = count_non_finite(X_train_processed)
test_non_finite = count_non_finite(X_test_processed)

assert X_train_processed.shape[0] == len(X_train)
assert X_test_processed.shape[0] == len(X_test)
assert X_train_processed.shape[1] == X_test_processed.shape[1]
assert X_train_processed.shape[1] == len(feature_names)
assert len(feature_names) == len(set(feature_names)), (
    "Transformed feature names are not unique."
)
assert train_non_finite == 0, "Training output contains NaN or infinity."
assert test_non_finite == 0, "Test output contains NaN or infinity."

print(f"Transformed features: {len(feature_names)}")
print(f"Non-finite values in training output: {train_non_finite}")
print(f"Non-finite values in test output: {test_non_finite}")
print("Processed feature validation passed.")

display(
    pd.DataFrame({"transformed_feature": feature_names}).head(20)
)

Transformed features: 116
Non-finite values in training output: 0
Non-finite values in test output: 0
Processed feature validation passed.


,transformed_feature
0,numerical__lead_time
1,numerical__arrival_date_year
2,numerical__arrival_date_week_number
3,numerical__arrival_date_day_of_month
4,numerical__stays_in_weekend_nights
5,numerical__stays_in_week_nights
6,numerical__adults
7,numerical__children
8,numerical__babies
9,numerical__previous_cancellations


### 9. Validate the Target, Split, and Leakage Exclusions

Verify that training and test rows do not overlap, every row belongs to one
partition, and the original `is_canceled` labels were preserved. Both classes
must remain present in each partition.

Also confirm that known outcome-related, post-booking, and personal fields
are absent from the selected predictors. This check follows the prediction
point defined for the project: initial reservation creation.

In [8]:
# Check that the split is complete and its original row indices do not overlap.
overlapping_rows = X_train.index.intersection(X_test.index)

assert len(overlapping_rows) == 0, "Training and test rows overlap."
assert len(X_train) + len(X_test) == len(df), "Some rows were lost or repeated."

# Confirm that y still matches the original target for each selected row.
assert y_train.equals(df.loc[X_train.index, "is_canceled"])
assert y_test.equals(df.loc[X_test.index, "is_canceled"])

assert set(y_train.unique()) == {0, 1}, "A target class is missing from training."
assert set(y_test.unique()) == {0, 1}, "A target class is missing from testing."

# These fields should not be predictors at the initial reservation point.
excluded_fields = {
    "reservation_status",
    "reservation_status_date",
    "assigned_room_type",
    "booking_changes",
    "days_in_waiting_list",
    "name",
    "email",
    "phone-number",
    "credit_card"
}

included_excluded_fields = excluded_fields.intersection(X.columns)
assert not included_excluded_fields, (
    f"Excluded fields found in predictors: {included_excluded_fields}"
)
assert "is_canceled" not in X.columns, "Target found among predictors."

print(f"Overlapping train/test rows: {len(overlapping_rows)}")
print("Original target labels preserved: Yes")
print(f"Excluded fields found in predictors: {len(included_excluded_fields)}")
print("Target, split, and leakage checks passed.")

Overlapping train/test rows: 0
Original target labels preserved: Yes
Excluded fields found in predictors: 0
Target, split, and leakage checks passed.


### 10. Create the Final Data-Quality Report

The final report records the input size, split sizes, class counts, transformed
feature count, and validation results. It provides a compact handoff for the
machine-learning stage.

A passing report confirms the checks performed in this notebook. It does not
establish the real-world recording time of every retained predictor; that
depends on information from the booking system.

In [9]:
REPORT_PATH = ROOT / "05_ML_Data_Preparation/data_quality_report.csv"

report_rows = [
    ("input_rows", len(df), "Rows in the selected feature dataset"),
    ("selected_predictors", len(selected_features), "Predictors before encoding"),
    ("training_rows", len(X_train), "Rows reserved for training"),
    ("test_rows", len(X_test), "Rows reserved for final testing"),
    ("training_not_canceled", int((y_train == 0).sum()), "Training target class 0"),
    ("training_canceled", int((y_train == 1).sum()), "Training target class 1"),
    ("test_not_canceled", int((y_test == 0).sum()), "Test target class 0"),
    ("test_canceled", int((y_test == 1).sum()), "Test target class 1"),
    ("transformed_features", len(feature_names), "Features after preprocessing"),
    ("overlapping_train_test_rows", len(overlapping_rows), "Expected: 0"),
    ("training_non_finite_values", train_non_finite, "Expected: 0"),
    ("test_non_finite_values", test_non_finite, "Expected: 0"),
    ("excluded_fields_in_predictors", len(included_excluded_fields), "Expected: 0"),
]

data_quality_report = pd.DataFrame(
    report_rows,
    columns=["metric", "value", "description"]
)

REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
data_quality_report.to_csv(REPORT_PATH, index=False)

display(data_quality_report)
print(f"Report saved to: {REPORT_PATH}")

,metric,value,description
0,input_rows,119205,Rows in the selected feature dataset
1,selected_predictors,32,Predictors before encoding
2,training_rows,95364,Rows reserved for training
3,test_rows,23841,Rows reserved for final testing
4,training_not_canceled,60008,Training target class 0
5,training_canceled,35356,Training target class 1
6,test_not_canceled,15002,Test target class 0
7,test_canceled,8839,Test target class 1
8,transformed_features,116,Features after preprocessing
9,overlapping_train_test_rows,0,Expected: 0


Report saved to: 05_ML_Data_Preparation\data_quality_report.csv


### 11. Conclusion and Modelling Handoff

The selected Hotel Booking dataset contains 119,205 records and 32 predictors.
The target, `is_canceled`, was separated from the predictors without changing
its labels. An 80/20 stratified split produced 95,364 training records and
23,841 test records, with the original class proportions preserved.

The preprocessing pipeline scales numerical features, passes binary indicators
through unchanged, one-hot encodes nominal categories, and groups infrequent
levels of high-cardinality categories. It was fitted on training data only.
The fitted pipeline transformed both partitions into matrices with 116
matching features.

Final checks found no non-finite transformed values, overlapping train/test
rows, or listed leakage fields among the predictors. The results are recorded
in `05_ML_Data_Preparation/data_quality_report.csv`.

The in-memory `X_train_processed`, `X_test_processed`, `y_train`, and `y_test`
are ready for the later modelling stage. That stage should keep the test set
for final evaluation and fit any model-selection steps using training data
only. The assumed availability of retained features at initial reservation
time should be verified against the booking system before deployment.